<a href="https://colab.research.google.com/github/Santiago-Echeverri-Arteaga/Fisica_Computacional_2/blob/master/curso_2026_2/03_redes_fundamentos/34_mlp_tensorflow.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg"
       alt="Abrir en Colab"/>
</a>

# Red feedforward con TensorFlow/Keras

**Pregunta guía:** ¿Cómo se traduce la derivación a una API entrenable?<br>
**Duración sugerida:** 4 horas.<br>
**Entorno:** CPU; datos incluidos o generados en memoria.

El orden de trabajo es siempre: problema → matemática → implementación
mínima → biblioteca → evaluación → interpretación física.


**Requiere:** runtime estándar de Colab con TensorFlow. Usaremos un
problema de clasificación no lineal, una partición fija de test y
validación para escoger anchura y tasa. `EarlyStopping` restaura los
pesos con menor pérdida de validación.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEMILLA = 42
tf.keras.utils.set_random_seed(SEMILLA)
X, y = make_moons(n_samples=1_600, noise=0.24, random_state=SEMILLA)
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEMILLA)
X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, test_size=0.2, stratify=y_dev, random_state=SEMILLA)
scaler = StandardScaler().fit(X_train)
X_train, X_val, X_test = map(scaler.transform, [X_train, X_val, X_test])
print("TensorFlow", tf.__version__, X_train.shape, X_val.shape, X_test.shape)


In [ ]:
def crear_mlp(unidades=32, tasa=1e-3):
    modelo = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(2,)),
            tf.keras.layers.Dense(unidades, activation="relu"),
            tf.keras.layers.Dense(unidades, activation="relu"),
            tf.keras.layers.Dense(1),  # logits, no probabilidades
        ]
    )
    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=tasa),
        loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
        metrics=[tf.keras.metrics.BinaryAccuracy(threshold=0.0, name="accuracy")],
    )
    return modelo

modelo_demo = crear_mlp()
modelo_demo.summary()


In [ ]:
configuraciones = [(16, 1e-3), (32, 1e-3), (64, 3e-4)]
filas, candidatos = [], []
for unidades, tasa in configuraciones:
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEMILLA)
    modelo = crear_mlp(unidades, tasa)
    parada = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=12, restore_best_weights=True
    )
    historia = modelo.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=150, batch_size=64, verbose=0, callbacks=[parada],
    )
    mejor_val = min(historia.history["val_loss"])
    filas.append({"unidades": unidades, "tasa": tasa, "val_loss": mejor_val, "épocas": len(historia.history["loss"])})
    candidatos.append((mejor_val, modelo, historia))
display(pd.DataFrame(filas).sort_values("val_loss"))


In [ ]:
_, mejor_modelo, historia = min(candidatos, key=lambda item: item[0])
logits = mejor_modelo.predict(X_test, verbose=0).ravel()
prob = tf.sigmoid(logits).numpy()
print("accuracy test:", accuracy_score(y_test, prob >= 0.5))
print("log loss test:", log_loss(y_test, prob))

plt.plot(historia.history["loss"], label="train")
plt.plot(historia.history["val_loss"], label="validation")
plt.xlabel("época"); plt.ylabel("BCE"); plt.legend(); plt.show()


## Un paso explícito con `GradientTape`

`fit` organiza el ciclo, pero la diferenciación puede verse directamente.
No ejecute este paso sobre el modelo final: es una demostración separada.


In [ ]:
demo = crear_mlp(16, 1e-3)
optimizador = demo.optimizer
x_batch = tf.convert_to_tensor(X_train[:64], dtype=tf.float32)
y_batch = tf.cast(y_train[:64, None], tf.float32)
with tf.GradientTape() as tape:
    logits = demo(x_batch, training=True)
    pérdida = tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(labels=y_batch, logits=logits))
grads = tape.gradient(pérdida, demo.trainable_variables)
optimizador.apply_gradients(zip(grads, demo.trainable_variables))
print("pérdida:", float(pérdida), "normas:", [float(tf.norm(g)) for g in grads])


**Ejercicios:** reproduzca con `tanh`; añada L2 y dropout por separado;
compare parámetros y tiempo; construya una activación propia de forma
vectorizada y verifique que TensorFlow pueda derivarla automáticamente.
